# Opening Leads — Demonstration

This notebook shows how well the **opening-lead engine** works. Standard leads
are encoded as declarative YAML rules under `bridge/data/leads/`, applied by a
small matcher in `bridge.play.leads_loader` — the same data-driven pattern as
the SAYC bidder.

Given the **holding** in the suit you've chosen to lead and the **contract
type** (`"suit"` = a trump contract, or `"nt"`), the engine returns *which
card to lead* and *what that card tells partner*.

- Base system: **ACBL SAYC** leads (4th best; A from A K x vs suits; low/high
  from three small; 2nd-highest from a long suit with no honour; top of
  touching/interior sequences).
- Opt-in variant: **3rd/5th best** (`system="three_five"`), applied **vs suit
  contracts only**.

In [1]:
from bridge.play import leads_loader as ll

rules = ll.load_rules()
meta = ll.load_meta()
print(meta["system"], "— version", meta["version"])
print(f"Loaded {len(rules)} lead rules; validator problems:",
      ll.validate_rules(rules) or "none")

Standard (ACBL SAYC) opening leads — version 0.1
Loaded 16 lead rules; validator problems: none


## 1. One holding, with the reasoning

`match_leads` returns every applicable rule ranked by priority; the top one is
the lead. Here is K Q J 6 against a suit contract.

In [2]:
def show_lead(holding, context, system="sayc"):
    cands = ll.match_leads(rules, holding, context, system)
    card = cands[0]["card"] if cands else "(none)"
    print(f"{holding}  vs {context}  [{system}]  ->  lead the {card}")
    for c in cands[:3]:
        print(f"    p{c['priority']:>3} {c['id']:<26} card={c['card']} "
              f"[{c['tag']}] — {c['shows'].split(chr(10))[0].strip()}")

show_lead("KQJ6", "suit")

KQJ6  vs suit  [sayc]  ->  lead the K
    p 90 lead_top_of_sequence       card=K [sayc] — Top of a 3+ card sequence (e.g. K from KQJ): promises the next touching cards, denies a higher honour.
    p 80 lead_two_honours_vs_suit   card=K [sayc] — Top of two touching honours vs a suit (e.g. K from KQxxx).
    p 70 lead_fourth_best           card=6 [sayc] — Fourth best from length and strength — partner applies the Rule of 11.


## 2. The standard table (SAYC), suit vs notrump

The classic card-combination table, generated straight from the rules. Note
where the contract changes the card (e.g. A K x, K Q x x x).

In [3]:
holdings = ["KQJ6", "QJT2", "JT9", "KJT4", "AQJ4", "KT9",
            "AK4", "AKQ5", "KQ842", "KQ9", "A8642", "K842",
            "8642", "Q72", "K72", "973", "A3", "5"]
print(f"{'holding':<8} {'vs suit':>8} {'vs NT':>7}")
for h in holdings:
    s = ll.choose_lead(rules, h, "suit") or "-"
    n = ll.choose_lead(rules, h, "nt") or "-"
    print(f"{h:<8} {s:>8} {n:>7}")

holding   vs suit   vs NT
KQJ6            K       K
QJT2            Q       Q
JT9             J       J
KJT4            J       J
AQJ4            A       Q
KT9             T       T
AK4             A       K
AKQ5            A       A
KQ842           K       4
KQ9             K       K
A8642           A       4
K842            2       2
8642            6       6
Q72             2       2
K72             2       2
973             3       9
A3              A       A
5               5       5


Each lead also carries the *message* it sends partner — the raw material
for reading the defence later (the inference engine reserves a `play=True`
channel for exactly this):

In [4]:
for h in ["KQJ6", "AK4", "KJ73", "A3", "973"]:
    top = ll.match_leads(rules, h, "suit")[0]
    print(f"{h:<6} -> {top['card']}  (signal: {top['signal']})")
    print(f"        {' '.join(top['shows'].split())}")

KQJ6   -> K  (signal: attitude)
        Top of a 3+ card sequence (e.g. K from KQJ): promises the next touching cards, denies a higher honour.
AK4    -> A  (signal: attitude)
        Lead the ace from any ace holding vs a suit — never underlead an ace against a trump contract.
KJ73   -> 3  (signal: count)
        Fourth best from length and strength — partner applies the Rule of 11.
A3     -> A  (signal: count)
        Top of a doubleton; the following high-low shows an even (two) count.
973    -> 3  (signal: count)
        Low from three small vs a suit.


## 3. SAYC 4th-best vs the 3rd/5th-best variant

3rd/5th best encodes *length parity* in the spot card: 3rd-highest from an
even-length suit, 5th (the lowest) from an odd-length suit. Per the chosen
agreement it applies **only vs suit contracts** — vs notrump the lead stays
4th best even when 3rd/5th is selected.

In [5]:
print(f"{'holding':<8} {'len':>3} {'4th best':>9} {'3rd/5th (suit)':>15} "
      f"{'3rd/5th (NT)':>13}")
for h in ["KJ73", "KJ863", "Q8654", "K8753"]:
    n = ll.holding_features(h)["length"]
    sayc = ll.choose_lead(rules, h, "suit", "sayc")
    tf_suit = ll.choose_lead(rules, h, "suit", "three_five")
    tf_nt = ll.choose_lead(rules, h, "nt", "three_five")
    print(f"{h:<8} {n:>3} {sayc:>9} {tf_suit:>15} {tf_nt:>13}")

holding  len  4th best  3rd/5th (suit)  3rd/5th (NT)
KJ73       4         3               7             3
KJ863      5         6               3             6
Q8654      5         5               4             5
K8753      5         5               3             5


## 4. Coverage & validity over random holdings

A blanket sanity check: deal many random single-suit holdings of every length
and confirm the engine (a) always returns a card and (b) only ever leads a
card actually held — in every context and system.

In [6]:
import random

RANKS = ll.RANKS  # "AKQJT98765432"
rng = random.Random(2026)
N = 20000
returned = held = 0
gaps = []

for _ in range(N):
    length = rng.randint(1, 13)
    holding = "".join(rng.sample(RANKS, length))
    for context in ("suit", "nt"):
        for system in ("sayc", "three_five"):
            card = ll.choose_lead(rules, holding, context, system)
            if card:
                returned += 1
                if card in holding:
                    held += 1
                else:
                    gaps.append((holding, context, system, card))
            else:
                gaps.append((holding, context, system, "(none)"))

trials = N * 2 * 2
print(f"Trials: {trials}")
print(f"Returned a card:        {returned}/{trials}  ({100*returned/trials:.1f}%)")
print(f"Card actually held:     {held}/{returned}  ({100*held/returned:.1f}%)")
print(f"Coverage/validity gaps: {len(gaps)}")

Trials: 80000
Returned a card:        80000/80000  (100.0%)
Card actually held:     80000/80000  (100.0%)
Coverage/validity gaps: 0


Every random holding produces a legal lead (a card the defender actually
holds), across both contract types and both lead systems — the rule base is
total over single-suit holdings.

## 5. Leading from a full hand

Given a hand and a chosen suit, pick the card. (Choosing *which* suit to lead
from the auction is a later, strategic layer; here we just show the card the
engine would play from each suit.)

In [7]:
hand = {"S": "KQJ4", "H": "A82", "D": "T9", "C": "8643"}
print("Hand:", hand, "  Contract: 4H by opponents (a suit contract)\n")
for suit, holding in hand.items():
    card = ll.choose_lead(rules, holding, "suit")
    top = ll.match_leads(rules, holding, "suit")[0]
    print(f"  {suit} {holding:<6} -> {card}   ({' '.join(top['shows'].split())})")

Hand: {'S': 'KQJ4', 'H': 'A82', 'D': 'T9', 'C': '8643'}   Contract: 4H by opponents (a suit contract)

  S KQJ4   -> K   (Top of a 3+ card sequence (e.g. K from KQJ): promises the next touching cards, denies a higher honour.)
  H A82    -> A   (Lead the ace from any ace holding vs a suit — never underlead an ace against a trump contract.)
  D T9     -> T   (Top of a doubleton; the following high-low shows an even (two) count.)
  C 8643   -> 6   (Second highest from a long suit with no honour (the SAYC systemic exception to fourth best).)


## Summary

- **Standard leads, data-driven.** The card-combination table is declarative
  YAML matched by priority — the same pattern as the SAYC bidder, and easy to
  extend or re-tag.
- **Grounded in the ACBL SAYC booklet** (cited per rule), with the
  combinations SAYC leaves implicit supplemented from cross-checked standard
  tables and tagged `standard`.
- **Two styles**: SAYC 4th-best by default, 3rd/5th-best opt-in (suit
  contracts only).
- **Total & sound** over random holdings: always a legal lead.
- **Each lead carries its message**, ready to feed play-based inferences.

Next (per `prompts/leads.md`): supplement rarer honour combinations, then add
the auction-driven *which suit to lead* strategy layer.